In [1]:
import sys
import subprocess
from pathlib import Path

import pandas as pd

# Directory paths
ROOT_DIR = Path.cwd()
SRC_DIR = ROOT_DIR / "src"
DATA_DIR = ROOT_DIR / "data"


def get_jsonl_files():
    return sorted(DATA_DIR.glob("*.jsonl"), key=lambda p: p.stat().st_mtime, reverse=True)

In [2]:
# Run the scraper script in src/
scraper_files = sorted(SRC_DIR.glob("*.py"))
if not scraper_files:
    raise FileNotFoundError("No Python files found in 'src/'.")

for script in scraper_files:
    result = subprocess.run(
        [sys.executable, str(script)],
        capture_output=True,
        text=True
    )
    print(f"Ran: {script.name}")
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"Error: {result.stderr}")

# Read the latest JSONL file as pandas DataFrame
jsonl_files = get_jsonl_files()
if not jsonl_files:
    raise FileNotFoundError("No JSONL file found in 'data/' after running scraper.")

jsonl_path = jsonl_files[0]
df = pd.read_json(jsonl_path, lines=True)

print(f"Loaded: {jsonl_path}")
df

Ran: edcs_scraper.py
[+] Connecting to EDCS API...
[+] Connected. Page size 500 works. Total in EDCS: 542,291

[lookup] No changes — using existing lookup file.

[+] Data files found. Checking for updates...
    Local monuments : 542,291
    EDCS total      : 542,291

[✓] No new records found — local data is up to date.
[✓] Local monuments : 542,291
[✓] EDCS total      : 542,291

[✓] JSONL  : /Users/sanojdoddapaneni/Documents/Projects/EDCS-Analytics/data/edcs_inscriptions.jsonl
[✓] TSV    : /Users/sanojdoddapaneni/Documents/Projects/EDCS-Analytics/data/edcs_inscriptions.tsv
[✓] Lookup : /Users/sanojdoddapaneni/Documents/Projects/EDCS-Analytics/data/edcs_lookup.json

Loaded: /Users/sanojdoddapaneni/Documents/Projects/EDCS-Analytics/data/edcs_inscriptions.jsonl


,record_id,edcs_id,inscription_index,province,place,latitude,longitude,material,material_en,not_before,not_after,inscription_text,language,category,category_en,image_urls
0,EDCS-00000001-0,EDCS-00000001,0,Noricum,Maribor / Marburg,46.555628,15.644770,lapis,stone,,,?] C(aius) Trebonius IIvir et praef(ectus) i(u...,la,"[tituli sepulcrales, viri, praenomen et nomen,...","[tomb inscriptions, men, first name and genti...",bilder/Mu/Muchar_1_p_398.jpg
1,EDCS-00000002-0,EDCS-00000002,0,NaN,NaN,NaN,NaN,lapis,stone,,,]ITSAMARVS[3] / [3]ETS TOVICTO[,la,[tituli sacri],[dedicatory inscriptions],bilder/No/Noelke-2021_00040.jpg
2,EDCS-00000003-0,EDCS-00000003,0,Germania inferior,Niederbachem / Wachtberg,50.646398,7.178584,lapis,stone,,,[I(ovi)] O(ptimo) M(aximo),la,[tituli sacri],[dedicatory inscriptions],bilder/Zi/Zimmermann-2022_19.jpg
3,EDCS-00000004-0,EDCS-00000004,0,Germania inferior,Niederbachem / Wachtberg,50.646398,7.178584,opus figlinae,ceramics,,,L(egio) I M(inervia),la,[tituli fabricationis],[manufacturer's inscriptions],
4,EDCS-00000005-0,EDCS-00000005,0,Venetia et Histria / Regio X,Este / Ateste,45.224007,11.659796,aes,bronze,,,C(aius) Artilius,la,"[sigilla impressa, praenomen et nomen]","[stamped inscriptions, first name and gentile...",bilder/Ar/ArchVen-2022-145.jpg | bilder/Ar/Arc...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
587882,EDCS-85701223-1,EDCS-85701223,1,Baetica,Almensilla,37.310001,-6.113052,lapis,stone,508,508,Hispalensis fa/mulus dei vixit / ann(os) plus ...,la,"[tituli sepulcrales, inscriptiones christianae...","[tomb inscriptions, christian inscriptions, me...",bilder/FE/FE_00898.jpg
587883,EDCS-85701224-0,EDCS-85701224,0,Hispania citerior,Banos de Rio Caldo,41.856406,-8.107254,opus figlinae,ceramics,,,[3]SCI / [3 O]gere(n)sibu[s,la,[tituli possessionis],[owner inscriptions],bilder/FE/FE_00899.jpg
587884,EDCS-85701225-0,EDCS-85701225,0,Moesia inferior,Mahmudia / Salsovia,45.086115,29.087574,aes,bronze,97,97,[Imp(erator) Ne]rva Ca[esar A]ugust[us pontife...,la,"[diplomata militaria, tria nomina, viri, Augus...","[military diplomas, tria nomina, men, emperor/...",
587885,EDCS-85701225-1,EDCS-85701225,1,Moesia inferior,Mahmudia / Salsovia,45.086115,29.087574,aes,bronze,97,97,quae est Lyciae et Pamphylia]e sub Iulio Mar[i...,la,"[diplomata militaria, tria nomina, viri, Augus...","[military diplomas, tria nomina, men, emperor/...",


In [3]:
list(df.columns)

['record_id',
 'edcs_id',
 'inscription_index',
 'province',
 'place',
 'latitude',
 'longitude',
 'material',
 'material_en',
 'not_before',
 'not_after',
 'inscription_text',
 'language',
 'category',
 'category_en',
 'image_urls']

In [9]:
missing_counts = df.isna().sum()
missing_counts[missing_counts > 0].sort_values(ascending=False)

latitude      20646
longitude     20646
province       3977
place          3977
not_after        35
not_before       22
dtype: int64

In [13]:
img = df['image_urls']

is_null = img.isna()
is_blank = img.astype('string').str.strip().eq('')
is_empty_list_obj = img.apply(lambda v: isinstance(v, list) and len(v) == 0)
is_empty_list_text = img.astype('string').str.strip().eq('[]')

missing_like = is_null | is_blank | is_empty_list_obj | is_empty_list_text

print(f"Rows: {len(df)}")
print(f"Null only (isna): {int(is_null.sum())}")
print(f"Blank strings: {int(is_blank.sum())}")
print(f"Empty list objects: {int(is_empty_list_obj.sum())}")
print(f"Text '[]': {int(is_empty_list_text.sum())}")
print(f"Missing-like total: {int(missing_like.sum())}")

df.loc[missing_like, ['edcs_id', 'image_urls']].head(10)

Rows: 587887
Null only (isna): 0
Blank strings: 463051
Empty list objects: 0
Text '[]': 0
Missing-like total: 463051


,edcs_id,image_urls
3,EDCS-00000004,
8,EDCS-00000009,
9,EDCS-00000010,
10,EDCS-00000010,
11,EDCS-00000011,
12,EDCS-00000012,
13,EDCS-00000013,
14,EDCS-00000014,
15,EDCS-00000015,
16,EDCS-00000016,


In [15]:
data = df.replace(r'^\s*$', pd.NA, regex=True)

In [19]:
null_info = data.isna().sum()
null_columns = pd.DataFrame({
    "null_count": null_info.astype("int64"),
    "rows": len(data),
})

null_columns["null_pct"] = (null_columns["null_count"] / len(data) * 100).round(2)

null_columns[null_columns["null_count"] > 0].sort_values("null_count", ascending=False)

,null_count,rows,null_pct
image_urls,463051,587887,78.77
not_after,368652,587887,62.71
not_before,368639,587887,62.71
material,81807,587887,13.92
material_en,81807,587887,13.92
latitude,20646,587887,3.51
longitude,20646,587887,3.51
province,3977,587887,0.68
place,3977,587887,0.68
inscription_text,140,587887,0.02


In [21]:
data.drop(columns=['image_urls'],inplace=True)

In [23]:
set_lan = set(df['language'])
set_lan

{'',
 '? ',
 '? , gr',
 'et',
 'et, gr',
 'et, la',
 'gr',
 'gr, gr',
 'gr, gr, la, la',
 'gr, he',
 'gr, he, la',
 'gr, la',
 'gr, la, pa',
 'gr, la, pu',
 'gr, pa',
 'gr, pu',
 'he',
 'he, la',
 'ib',
 'ib, la',
 'la',
 'la, la',
 'la, li',
 'la, os',
 'la, pa',
 'la, pu',
 'la, ve',
 'le',
 'os',
 'pa',
 'pu',
 'ra',
 'sa',
 've'}